# SGRAC unit mesh: local slip-gradient prototype

## Introduction and conventions

This notebook is a pedagogical prototype for developing the local slip-gradient strategy on a small triangular mesh.

The current goal is to prepare a simplified Python version of the Fortran `NTON` edge structure so that a later step can loop edge by edge and store tangential and normal slip-gradient components directly on each canonical edge.

Fixed conventions for the future edge loop:

```text
+t = smaller node index -> larger node index
local third node of cellonedge(1) will be placed at y < 0
local third node of cellonedge(2) will be placed at y > 0
+n = cellonedge(1) -> cellonedge(2)
ds = slip(cellonedge(2)) - slip(cellonedge(1))
```

The mesh points are already in the original `(x, y)` plane, so no trilateration is used in this preparation step. Boundary-edge gradients remain initialized to zero and will not be used.


## Read the mesh and slip values

The reader below is intentionally small and specific to the SGRAC-style legacy ASCII VTK `POLYDATA` unit mesh used in this prototype. It reads triangular `POLYGONS` connectivity and one scalar `slip` value per cell.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


vtk_file = Path("4cellmesh_sgrac_polydata_slip.vtk")


def read_sgrac_polydata_vtk(path):
    """Read a minimal SGRAC-style legacy ASCII VTK POLYDATA triangle mesh.

    Expected structure:
    - POINTS <npoints> <type>
    - POLYGONS <ncells> <total_size>
    - optional CELL_DATA <ncells>
      SCALARS slip ...
      LOOKUP_TABLE default
      <one scalar per cell>

    This parser is intentionally small and pedagogical.
    It is not a general VTK reader.
    """
    path = Path(path)
    lines = path.read_text().splitlines()

    points = []
    cells = []
    slip = None

    i = 0
    while i < len(lines):
        parts = lines[i].strip().split()

        if not parts:
            i += 1
            continue

        if parts[0].upper() == "POINTS":
            npoints = int(parts[1])
            values = []
            i += 1
            while len(values) < 3 * npoints:
                values.extend(float(x) for x in lines[i].strip().split())
                i += 1
            points = np.array(values, dtype=float).reshape(npoints, 3)
            continue

        if parts[0].upper() == "POLYGONS":
            ncells = int(parts[1])
            i += 1
            for _ in range(ncells):
                row = [int(x) for x in lines[i].strip().split()]
                if row[0] != 3:
                    raise ValueError("Only triangular POLYGONS are supported.")
                cells.append(row[1:4])
                i += 1
            cells = np.array(cells, dtype=int)
            continue

        if parts[0].upper() == "CELL_DATA":
            ncells = int(parts[1])
            i += 1

            # Look for the SCALARS slip block.
            while i < len(lines):
                p = lines[i].strip().split()
                if len(p) >= 2 and p[0].upper() == "SCALARS" and p[1] == "slip":
                    i += 1
                    if lines[i].strip().upper().startswith("LOOKUP_TABLE"):
                        i += 1
                    values = []
                    while len(values) < ncells and i < len(lines):
                        if lines[i].strip():
                            values.extend(float(x) for x in lines[i].strip().split())
                        i += 1
                    slip = np.array(values[:ncells], dtype=float)
                    break
                i += 1
            continue

        i += 1

    if len(points) == 0:
        raise ValueError("No POINTS block found.")
    if len(cells) == 0:
        raise ValueError("No POLYGONS block found.")
    if slip is None:
        slip = np.full(len(cells), np.nan)

    return points, cells, slip


points, cells, slip = read_sgrac_polydata_vtk(vtk_file)

print(f"Number of points: {len(points)}")
print(f"Number of triangular cells: {len(cells)}")
print("Slip values:", slip)


## Plot the raw mesh

The first plot shows only the raw triangular mesh: triangle edges, point labels, cell labels, axes, and grid.


In [ ]:
def cell_barycentre(points, cell_nodes):
    """Return the x-y barycentre of one triangular cell."""
    return points[np.asarray(cell_nodes, dtype=int), :2].mean(axis=0)


def triangle_barycentres(points, cells):
    """Return the barycentre of each triangular cell."""
    return np.array([cell_barycentre(points, tri) for tri in cells])


def draw_triangular_mesh(ax, points, cells, linewidth=1.5):
    """Draw only triangle edges in the existing x-y plane."""
    xy = points[:, :2]
    for tri in cells:
        tri_xy = xy[tri]
        closed = np.vstack([tri_xy, tri_xy[0]])
        ax.plot(closed[:, 0], closed[:, 1], linewidth=linewidth, color="C0")


def finish_mesh_axes(ax, title):
    """Apply common axes styling for the small pedagogical mesh plots."""
    ax.axhline(0.0, linewidth=0.8, color="0.5")
    ax.axvline(0.0, linewidth=0.8, color="0.5")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, linewidth=0.5)


def add_point_labels(ax, points):
    """Label mesh points as P0, P1, ..."""
    xy = points[:, :2]
    ax.scatter(xy[:, 0], xy[:, 1], zorder=3)
    for ip, (x, y) in enumerate(xy):
        ax.text(x, y, f"  P{ip}", ha="left", va="bottom", fontsize=10)


def add_cell_labels(ax, points, cells):
    """Label triangular cells as C0, C1, ..."""
    bary = triangle_barycentres(points, cells)
    for icell, (x, y) in enumerate(bary):
        ax.text(x, y, f"C{icell}", ha="center", va="center", fontsize=10)


def plot_raw_mesh(points, cells):
    """Plot only the raw mesh topology: points, cells, axes, and grid."""
    fig, ax = plt.subplots(figsize=(7, 6))
    draw_triangular_mesh(ax, points, cells)
    add_point_labels(ax, points)
    add_cell_labels(ax, points, cells)
    finish_mesh_axes(ax, "Raw triangular mesh")
    return fig, ax


fig, ax = plot_raw_mesh(points, cells)
plt.show()


## Build standard NTON connectivity

In the Fortran topology routines, `NTON` stores canonical node-to-node edge connectivity. For this notebook prototype we reconstruct the same idea directly from the triangular `POLYGONS` connectivity.

Each triangle contributes three local edges. To avoid storing the same edge twice with opposite orientations, every edge is converted to the canonical pair:

```text
(min_node, max_node)
```

For each canonical edge we store the two node indices, the Euclidean length in the existing `(x, y)` plane, and the adjacent cell list in discovery order. An internal edge has two adjacent cells. A boundary edge has one adjacent cell and is conceptually equivalent to `cellonedge(2) = 0` on the Fortran side, but the Python representation keeps the missing second cell as `None` to avoid confusing it with Python cell index `0`.


In [ ]:
from dataclasses import dataclass, field


@dataclass
class CanonicalEdge:
    """Simplified Python equivalent of one Fortran NTON edge entry."""

    node1: int
    node2: int
    length: float
    adjacent_cells: list[int] = field(default_factory=list)
    sgt: float = 0.0
    sgn: float = 0.0

    @property
    def kind(self):
        if len(self.adjacent_cells) == 1:
            return "boundary"
        if len(self.adjacent_cells) == 2:
            return "internal"
        return "non-manifold"

    @property
    def cell1(self):
        """First adjacent Python cell index, preserving discovery order."""
        return self.adjacent_cells[0]

    @property
    def cell2(self):
        """Second adjacent Python cell index, or None for a boundary edge."""
        if len(self.adjacent_cells) < 2:
            return None
        return self.adjacent_cells[1]

    @property
    def cellonedge1_fortran(self):
        """Conceptual one-based Fortran cellonedge(1) slot."""
        return self.cell1 + 1

    @property
    def cellonedge2_fortran(self):
        """Conceptual Fortran cellonedge(2) slot; 0 means boundary sentinel."""
        return 0 if self.cell2 is None else self.cell2 + 1


def triangle_edges(tri):
    """Return the three local edges of a triangular cell."""
    return [
        (tri[0], tri[1]),
        (tri[1], tri[2]),
        (tri[2], tri[0]),
    ]


def canonical_edge(a, b):
    """Store an edge once, independent of local triangle orientation."""
    return (min(a, b), max(a, b))


def reconstruct_nton_edges(points, cells):
    """Build canonical node-to-node connectivity from triangular cells."""
    xy = points[:, :2]
    edges = {}

    for icell, tri in enumerate(cells):
        for a, b in triangle_edges(tri):
            n1, n2 = canonical_edge(int(a), int(b))
            key = (n1, n2)

            if key not in edges:
                length = float(np.linalg.norm(xy[n2] - xy[n1]))
                edges[key] = CanonicalEdge(
                    node1=n1,
                    node2=n2,
                    length=length,
                )

            edges[key].adjacent_cells.append(icell)

    return dict(sorted(edges.items()))


nton_edges = reconstruct_nton_edges(points, cells)

print(f"Number of canonical edges: {len(nton_edges)}")
print("edge_id  edge      length      adjacent_cells  kind")
print("-------  --------  ----------  --------------  --------")

for iedge, ((n1, n2), edge) in enumerate(nton_edges.items()):
    print(
        f"E{iedge:<6} ({n1:>1}, {n2:>1})  "
        f"{edge.length:10.4f}  "
        f"{edge.adjacent_cells!s:<14}  "
        f"{edge.kind}"
    )

# Sanity checks matching the simplified manifold expectation.
assert len(nton_edges) == len(set(nton_edges)), "Each canonical edge should appear once."
for (n1, n2), edge in nton_edges.items():
    assert n1 < n2, "Canonical orientation must be (min_node, max_node)."
    assert edge.sgt == 0.0
    assert edge.sgn == 0.0
    if edge.kind == "internal":
        assert len(edge.adjacent_cells) == 2
    elif edge.kind == "boundary":
        assert len(edge.adjacent_cells) == 1
        assert edge.cell2 is None
        assert edge.cellonedge2_fortran == 0
    else:
        raise ValueError(f"Unexpected non-manifold edge {(n1, n2)}: {edge.adjacent_cells}")

assert nton_edges[(0, 1)].adjacent_cells == [1, 3]
assert nton_edges[(0, 2)].adjacent_cells == [0, 1]
assert nton_edges[(1, 2)].adjacent_cells == [1, 2]


## Plot NTON connectivity

This plot keeps canonical edge connectivity separate from slip-gradient storage. Edge labels include the canonical edge id and the boundary/internal classification.


In [ ]:
def plot_nton_edges(points, cells, edges, show_point_ids=False, show_cell_ids=False):
    """Plot canonical NTON-style edges without slip or barycentre labels."""
    xy = points[:, :2]
    fig, ax = plt.subplots(figsize=(7, 6))
    draw_triangular_mesh(ax, points, cells)

    if show_point_ids:
        add_point_labels(ax, points)
    else:
        ax.scatter(xy[:, 0], xy[:, 1], zorder=3, color="C0")

    if show_cell_ids:
        add_cell_labels(ax, points, cells)

    for iedge, ((n1, n2), edge) in enumerate(edges.items()):
        midpoint = 0.5 * (xy[n1] + xy[n2])
        label_color = "#e8f2ff" if edge.kind == "internal" else "#fff3d6"
        ax.text(
            midpoint[0],
            midpoint[1],
            f"E{iedge}\n({n1},{n2})\n{edge.kind}",
            ha="center",
            va="center",
            fontsize=9,
            bbox={
                "boxstyle": "round,pad=0.2",
                "facecolor": label_color,
                "edgecolor": "0.5",
                "alpha": 0.9,
            },
        )

    finish_mesh_axes(ax, "NTON canonical edge connectivity")
    return fig, ax


fig, ax = plot_nton_edges(points, cells, nton_edges)
plt.show()


## Prepare NTON for slip-gradient storage

The edge dictionary is now ready to carry future slip-gradient components directly on each canonical edge. The authoritative Python topology is still `adjacent_cells`, with zero-based cell indices and discovery order preserved.

Fortran-style slots are only a documented conceptual mapping:

```text
cellonedge(1) = edge.cellonedge1_fortran
cellonedge(2) = edge.cellonedge2_fortran
```

For an internal edge, `cellonedge(2)` maps to the second adjacent Python cell plus one. For a boundary edge, `cellonedge(2) = 0` is the Fortran sentinel. In Python, the missing second cell remains `None`, so Python cell index `0` is never confused with a boundary sentinel.

The scalar slip-gradient components are initialized but not computed in this step:

```text
sgt = 0.0   tangential slip-gradient component along +t
sgn = 0.0   normal slip-gradient component along +n
```

The helper `cell_barycentre(points, cell_nodes)` is kept because the future edge loop will need the barycentre of each adjacent cell locally, without making barycentres a separate algorithmic phase here.


In [ ]:
print("edge_id  edge      length      cell1_py  cell2_py  kind      sgt       sgn")
print("-------  --------  ----------  --------  --------  --------  --------  --------")

for iedge, ((n1, n2), edge) in enumerate(nton_edges.items()):
    cell2_label = "None" if edge.cell2 is None else str(edge.cell2)
    print(
        f"E{iedge:<6} ({n1:>1}, {n2:>1})  "
        f"{edge.length:10.4f}  "
        f"{edge.cell1:8d}  "
        f"{cell2_label:>8}  "
        f"{edge.kind:<8}  "
        f"{edge.sgt:8.3f}  "
        f"{edge.sgn:8.3f}"
    )

# This section prepares storage only. The gradient components remain untouched.
assert all(edge.sgt == 0.0 and edge.sgn == 0.0 for edge in nton_edges.values())
assert len(nton_edges) == 9


## Edge loop: local frame and slip-gradient components

This section loops over every canonical `NTON` edge. Boundary edges keep their initialized values `sgt = 0.0` and `sgn = 0.0`.

For each internal edge, the two adjacent cells are used in their stored discovery order. The canonical node direction defines `+t`, with `node1` at local `(0, 0)` and `node2` at local `(L, 0)`. The local normal orientation is then chosen by reflection, if needed, so that the third node of `cellonedge(1)` lies at `y < 0` and the third node of `cellonedge(2)` lies at `y > 0`. The adjacent-cell order is never changed.

The scalar slip is constant in each cell and attached to the cell barycentre. For the two adjacent barycentres `B1` and `B2`:

```text
dB = B2 - B1
ds = slip(cellonedge(2)) - slip(cellonedge(1))
G  = ds * dB / dot(dB, dB)
sgt = Gx
sgn = Gy
```

This is still a flat `(x, y)` notebook prototype. No geodesic trilateration, cell-gradient averaging, or boundary-edge gradient computation is performed here.


In [ ]:
def third_node_of_triangle(tri, edge_nodes):
    """Return the node of triangular cell `tri` that is not on `edge_nodes`."""
    edge_node_set = set(edge_nodes)
    third_nodes = [int(node) for node in tri if int(node) not in edge_node_set]
    if len(third_nodes) != 1:
        raise ValueError(f"Triangle {tri} does not contain exactly one third node for edge {edge_nodes}.")
    return third_nodes[0]


def edge_local_transform(points, node1, node2):
    """Return a function mapping global x-y points into the canonical edge frame."""
    xy = points[:, :2]
    p1 = xy[node1]
    p2 = xy[node2]
    edge_vector = p2 - p1
    length = float(np.linalg.norm(edge_vector))
    if length <= 0.0:
        raise ValueError(f"Edge ({node1}, {node2}) has non-positive length.")

    tangent = edge_vector / length
    normal = np.array([-tangent[1], tangent[0]])

    def to_local(point_xy):
        relative = np.asarray(point_xy, dtype=float) - p1
        return np.array([np.dot(relative, tangent), np.dot(relative, normal)])

    return to_local, length


def oriented_local_edge_geometry(points, cells, edge):
    """Build local geometry for one internal edge without changing adjacent-cell order."""
    if edge.kind != "internal" or len(edge.adjacent_cells) != 2:
        raise ValueError("Local two-cell geometry is defined only for internal edges.")

    cell1, cell2 = edge.adjacent_cells
    tri1 = cells[cell1]
    tri2 = cells[cell2]
    edge_nodes = (edge.node1, edge.node2)

    third1 = third_node_of_triangle(tri1, edge_nodes)
    third2 = third_node_of_triangle(tri2, edge_nodes)
    to_local, length = edge_local_transform(points, edge.node1, edge.node2)

    local_node1 = to_local(points[edge.node1, :2])
    local_node2 = to_local(points[edge.node2, :2])
    local_third1 = to_local(points[third1, :2])
    local_third2 = to_local(points[third2, :2])

    if not (local_third1[1] * local_third2[1] < 0.0):
        raise ValueError(
            f"Third nodes for edge ({edge.node1}, {edge.node2}) are not on opposite sides: "
            f"y1={local_third1[1]}, y2={local_third2[1]}"
        )

    # Keep cell1/cell2 fixed. If needed, reflect the local normal coordinate
    # so cell1 is below the edge and cell2 is above it.
    normal_sign = -1.0 if local_third1[1] > 0.0 else 1.0

    def orient(local_point):
        oriented = np.array(local_point, dtype=float)
        oriented[1] *= normal_sign
        return oriented

    local_node1 = orient(local_node1)
    local_node2 = orient(local_node2)
    local_third1 = orient(local_third1)
    local_third2 = orient(local_third2)

    if not (local_third1[1] < 0.0 and local_third2[1] > 0.0):
        raise ValueError(
            f"Could not orient edge ({edge.node1}, {edge.node2}) with cell1 below and cell2 above."
        )

    local_tri1 = np.array([local_node1, local_node2, local_third1])
    local_tri2 = np.array([local_node1, local_node2, local_third2])
    local_bary1 = local_tri1.mean(axis=0)
    local_bary2 = local_tri2.mean(axis=0)

    if not (local_bary1[1] < 0.0 and local_bary2[1] > 0.0):
        raise ValueError(
            f"Barycentres for edge ({edge.node1}, {edge.node2}) do not respect the local normal orientation."
        )

    return {
        "cell1": cell1,
        "cell2": cell2,
        "third1": third1,
        "third2": third2,
        "length": length,
        "node1": local_node1,
        "node2": local_node2,
        "third1_xy": local_third1,
        "third2_xy": local_third2,
        "tri1_xy": local_tri1,
        "tri2_xy": local_tri2,
        "bary1": local_bary1,
        "bary2": local_bary2,
    }


def compute_edge_slip_gradients(points, cells, slip, edges):
    """Compute and store sgt/sgn for internal canonical edges only."""
    diagnostics = []

    for iedge, ((n1, n2), edge) in enumerate(edges.items()):
        if edge.length <= 0.0:
            raise ValueError(f"Edge E{iedge} ({n1}, {n2}) has non-positive length.")

        if edge.kind == "boundary":
            edge.sgt = 0.0
            edge.sgn = 0.0
            diagnostics.append(
                {
                    "iedge": iedge,
                    "nodes": (n1, n2),
                    "cells": list(edge.adjacent_cells),
                    "kind": edge.kind,
                    "ds": 0.0,
                    "dBx": 0.0,
                    "dBy": 0.0,
                    "sgt": edge.sgt,
                    "sgn": edge.sgn,
                    "geometry": None,
                }
            )
            continue

        if edge.kind != "internal" or len(edge.adjacent_cells) != 2:
            raise ValueError(f"Edge E{iedge} ({n1}, {n2}) is not a valid internal manifold edge.")

        geometry = oriented_local_edge_geometry(points, cells, edge)
        cell1 = geometry["cell1"]
        cell2 = geometry["cell2"]
        s1 = float(slip[cell1])
        s2 = float(slip[cell2])
        ds = s2 - s1

        dB = geometry["bary2"] - geometry["bary1"]
        denominator = float(np.dot(dB, dB))
        if denominator <= 1.0e-14:
            raise ValueError(f"Barycentres for edge E{iedge} ({n1}, {n2}) are too close.")

        G = ds * dB / denominator
        edge.sgt = float(G[0])
        edge.sgn = float(G[1])

        diagnostics.append(
            {
                "iedge": iedge,
                "nodes": (n1, n2),
                "cells": list(edge.adjacent_cells),
                "kind": edge.kind,
                "ds": ds,
                "dBx": float(dB[0]),
                "dBy": float(dB[1]),
                "sgt": edge.sgt,
                "sgn": edge.sgn,
                "geometry": geometry,
                "s1": s1,
                "s2": s2,
            }
        )

    return diagnostics


edge_gradient_diagnostics = compute_edge_slip_gradients(points, cells, slip, nton_edges)

print("edge_id  edge      cells       ds        dBx       dBy       sgt       sgn       kind")
print("-------  --------  ----------  --------  --------  --------  --------  --------  --------")

for row in edge_gradient_diagnostics:
    n1, n2 = row["nodes"]
    print(
        f"E{row['iedge']:<6} ({n1:>1}, {n2:>1})  "
        f"{str(row['cells']):<10}  "
        f"{row['ds']:8.4f}  "
        f"{row['dBx']:8.4f}  "
        f"{row['dBy']:8.4f}  "
        f"{row['sgt']:8.4f}  "
        f"{row['sgn']:8.4f}  "
        f"{row['kind']}"
    )

internal_rows = [row for row in edge_gradient_diagnostics if row["kind"] == "internal"]
boundary_rows = [row for row in edge_gradient_diagnostics if row["kind"] == "boundary"]

assert len(edge_gradient_diagnostics) == len(nton_edges) == 9
assert len(internal_rows) == 3
assert all(row["sgt"] == 0.0 and row["sgn"] == 0.0 for row in boundary_rows)
assert all(row["geometry"]["bary1"][1] < 0.0 for row in internal_rows)
assert all(row["geometry"]["bary2"][1] > 0.0 for row in internal_rows)


In [ ]:
def plot_internal_edge_gradient(row):
    """Draw one internal edge in its oriented local frame."""
    geometry = row["geometry"]
    if geometry is None:
        return None, None

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    tri1 = np.vstack([geometry["tri1_xy"], geometry["tri1_xy"][0]])
    tri2 = np.vstack([geometry["tri2_xy"], geometry["tri2_xy"][0]])
    ax.plot(tri1[:, 0], tri1[:, 1], color="C1", linewidth=1.5, label=f"cell {geometry['cell1']}")
    ax.plot(tri2[:, 0], tri2[:, 1], color="C2", linewidth=1.5, label=f"cell {geometry['cell2']}")

    p1 = geometry["node1"]
    p2 = geometry["node2"]
    ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], color="black", zorder=4)
    ax.text(p1[0], p1[1], f"  P{row['nodes'][0]}", va="bottom")
    ax.text(p2[0], p2[1], f"  P{row['nodes'][1]}", va="bottom")

    b1 = geometry["bary1"]
    b2 = geometry["bary2"]
    ax.scatter([b1[0]], [b1[1]], color="C1", marker="x", s=80, linewidths=2.0, zorder=5)
    ax.scatter([b2[0]], [b2[1]], color="C2", marker="x", s=80, linewidths=2.0, zorder=5)
    ax.text(b1[0], b1[1], f"  B1\ns1={row['s1']:g}", color="C1", va="top")
    ax.text(b2[0], b2[1], f"  B2\ns2={row['s2']:g}", color="C2", va="bottom")

    ax.annotate(
        "",
        xy=b2,
        xytext=b1,
        arrowprops={"arrowstyle": "->", "color": "0.25", "linewidth": 1.8},
    )
    ax.text(
        0.5 * (b1[0] + b2[0]),
        0.5 * (b1[1] + b2[1]),
        "  B1 -> B2",
        color="0.25",
    )

    ax.text(
        0.02,
        0.98,
        f"E{row['iedge']} {row['nodes']}\n"
        f"ds={row['ds']:.4g}\n"
        f"sgt={row['sgt']:.4g}\n"
        f"sgn={row['sgn']:.4g}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "0.6", "alpha": 0.95},
    )

    ax.axhline(0.0, linewidth=0.8, color="0.5")
    ax.axvline(0.0, linewidth=0.8, color="0.5")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("local t  (+ from smaller node to larger node)")
    ax.set_ylabel("local n  (+ from cell1 to cell2)")
    ax.set_title(f"Internal edge E{row['iedge']}: local slip-gradient frame")
    ax.grid(True, linewidth=0.5)
    ax.legend(loc="best")
    return fig, ax


for row in edge_gradient_diagnostics:
    if row["kind"] == "internal":
        fig, ax = plot_internal_edge_gradient(row)
        plt.show()


## Cell slip-gradient reconstruction from edge vectors

Each internal edge now stores two scalar components in its own canonical local frame:

```text
g_edge = sgt * t + sgn * n
```

where `t` is the canonical unit tangent from the smaller node index to the larger node index, and `n` points from `cellonedge(1)` toward `cellonedge(2)`.

To reconstruct one cell slip-gradient vector, each edge contribution must first be expressed in a common frame. In this notebook the mesh is already flat in the global `(x, y)` plane, so the global Cartesian frame is the common cell frame.

The first reconstruction rule is the unweighted vector average of valid internal-edge vectors:

```text
g_cell = sum(g_edge_i) / Nvalid
```

Boundary edges are skipped because their stored zero values mean "not computed", not a physical zero gradient. The first demonstration uses the central cell `C1`, whose three edges are internal in this unit mesh.

A future alternative could use weights proportional to `1 / dbary`, which would require retaining or recomputing the barycentre-to-barycentre distance for each edge.


In [ ]:
def canonical_edges_of_cell(cell_id, cells):
    """Return the three canonical node pairs of one triangular cell."""
    tri = cells[cell_id]
    return [canonical_edge(int(a), int(b)) for a, b in triangle_edges(tri)]


def canonical_tangent_global(points, edge):
    """Return the canonical unit tangent in the global x-y frame."""
    xy = points[:, :2]
    vector = xy[edge.node2] - xy[edge.node1]
    length = float(np.linalg.norm(vector))
    if length <= 0.0:
        raise ValueError(f"Edge ({edge.node1}, {edge.node2}) has non-positive length.")
    return vector / length


def canonical_normal_global(points, cells, edge):
    """Return the canonical unit normal in the global x-y frame.

    The sign is chosen so dot(B2 - B1, n) > 0, where B1 and B2 are the
    barycentres of cellonedge(1) and cellonedge(2), respectively.
    """
    if edge.kind != "internal" or len(edge.adjacent_cells) != 2:
        raise ValueError("A canonical normal is defined here only for internal edges.")

    tangent = canonical_tangent_global(points, edge)
    normal = np.array([-tangent[1], tangent[0]])

    cell1, cell2 = edge.adjacent_cells
    bary1 = cell_barycentre(points, cells[cell1])
    bary2 = cell_barycentre(points, cells[cell2])
    dB = bary2 - bary1

    if np.dot(dB, normal) < 0.0:
        normal = -normal

    projection = float(np.dot(dB, normal))
    if projection <= 0.0:
        raise ValueError(
            f"Could not orient normal from cell {cell1} to cell {cell2} "
            f"for edge ({edge.node1}, {edge.node2})."
        )

    return normal


def cell_role_for_edge(cell_id, edge):
    """Return whether a cell is cellonedge(1) or cellonedge(2)."""
    if edge.kind != "internal":
        return "boundary"
    if cell_id == edge.cell1:
        return "cellonedge(1)"
    if cell_id == edge.cell2:
        return "cellonedge(2)"
    raise ValueError(f"Cell {cell_id} is not adjacent to edge ({edge.node1}, {edge.node2}).")


def compute_cell_slip_gradient(cell_id, points, cells, nton_edges):
    """Reconstruct one cell slip-gradient vector from valid internal edge vectors."""
    edge_id_by_pair = {pair: iedge for iedge, pair in enumerate(nton_edges.keys())}
    cell_edge_pairs = canonical_edges_of_cell(cell_id, cells)

    if len(cell_edge_pairs) != 3:
        raise ValueError(f"Cell {cell_id} does not have exactly three edges.")

    contributions = []

    for pair in cell_edge_pairs:
        if pair not in nton_edges:
            raise KeyError(f"Cell {cell_id} edge {pair} was not found in NTON.")

        edge = nton_edges[pair]
        iedge = edge_id_by_pair[pair]

        if edge.kind == "boundary":
            continue

        if edge.kind != "internal" or len(edge.adjacent_cells) != 2:
            raise ValueError(f"Edge E{iedge} {pair} is not a valid internal edge.")

        tangent = canonical_tangent_global(points, edge)
        normal = canonical_normal_global(points, cells, edge)
        cell1, cell2 = edge.adjacent_cells
        bary1 = cell_barycentre(points, cells[cell1])
        bary2 = cell_barycentre(points, cells[cell2])

        if np.dot(bary2 - bary1, normal) <= 0.0:
            raise ValueError(f"Internal edge normal for E{iedge} {pair} does not point from cell1 to cell2.")

        gradient = edge.sgt * tangent + edge.sgn * normal
        if not np.all(np.isfinite(gradient)):
            raise ValueError(f"Non-finite reconstructed gradient for edge E{iedge} {pair}.")

        contributions.append(
            {
                "iedge": iedge,
                "nodes": pair,
                "edge": edge,
                "role": cell_role_for_edge(cell_id, edge),
                "tangent": tangent,
                "normal": normal,
                "sgt": edge.sgt,
                "sgn": edge.sgn,
                "gradient": gradient,
            }
        )

    if len(contributions) == 0:
        raise ValueError(f"Cell {cell_id} has no valid internal edge gradient vectors.")

    vectors = np.array([item["gradient"] for item in contributions])
    vector_sum = vectors.sum(axis=0)
    cell_gradient = vector_sum / len(contributions)

    if not np.all(np.isfinite(cell_gradient)):
        raise ValueError(f"Non-finite reconstructed cell gradient for cell {cell_id}.")

    return {
        "cell_id": cell_id,
        "cell_gradient": cell_gradient,
        "edge_gradients": vectors,
        "edge_ids": [item["iedge"] for item in contributions],
        "contributions": contributions,
        "vector_sum": vector_sum,
        "nvalid": len(contributions),
    }


selected_cell_id = 1
selected_cell_result = compute_cell_slip_gradient(selected_cell_id, points, cells, nton_edges)

# First notebook unit test: C1 has three valid internal edge contributions.
assert len(canonical_edges_of_cell(selected_cell_id, cells)) == 3
assert selected_cell_result["nvalid"] == 3
assert all(nton_edges[pair].kind == "internal" for pair in canonical_edges_of_cell(selected_cell_id, cells))

print(f"Selected cell: C{selected_cell_id}")
print("edge_id  edge      role           tx        ty        nx        ny        sgt       sgn       gx        gy")
print("-------  --------  -------------  --------  --------  --------  --------  --------  --------  --------  --------")

for item in selected_cell_result["contributions"]:
    n1, n2 = item["nodes"]
    t = item["tangent"]
    n = item["normal"]
    g = item["gradient"]
    print(
        f"E{item['iedge']:<6} ({n1:>1}, {n2:>1})  "
        f"{item['role']:<13}  "
        f"{t[0]:8.4f}  {t[1]:8.4f}  "
        f"{n[0]:8.4f}  {n[1]:8.4f}  "
        f"{item['sgt']:8.4f}  {item['sgn']:8.4f}  "
        f"{g[0]:8.4f}  {g[1]:8.4f}"
    )

print("\nReconstructed edge-gradient vectors:")
for item in selected_cell_result["contributions"]:
    g = item["gradient"]
    print(f"  E{item['iedge']}: [{g[0]:.6f}, {g[1]:.6f}]")

vector_sum = selected_cell_result["vector_sum"]
cell_gradient = selected_cell_result["cell_gradient"]
gradient_magnitude = float(np.linalg.norm(cell_gradient))
gradient_angle_deg = float(np.degrees(np.arctan2(cell_gradient[1], cell_gradient[0])))

print(f"\nVector sum: [{vector_sum[0]:.6f}, {vector_sum[1]:.6f}]")
print(f"Unweighted vector average: [{cell_gradient[0]:.6f}, {cell_gradient[1]:.6f}]")
print(f"Magnitude: {gradient_magnitude:.6f}")
print(f"Direction angle from global +x: {gradient_angle_deg:.3f} degrees")


In [ ]:
def plot_cell_slip_gradient(points, cells, cell_id, cell_result):
    """Plot reconstructed edge vectors and the averaged cell gradient for one cell."""
    tri_nodes = cells[cell_id]
    tri_xy = points[tri_nodes, :2]
    closed = np.vstack([tri_xy, tri_xy[0]])
    bary = cell_barycentre(points, tri_nodes)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(closed[:, 0], closed[:, 1], color="C0", linewidth=1.8, label=f"cell C{cell_id}")
    ax.scatter(tri_xy[:, 0], tri_xy[:, 1], color="C0", zorder=4)

    for node, (x, y) in zip(tri_nodes, tri_xy):
        ax.text(x, y, f"  P{int(node)}", va="bottom")

    ax.scatter([bary[0]], [bary[1]], marker="x", s=90, linewidths=2.0, color="black", zorder=5)
    ax.text(bary[0], bary[1], f"  B{cell_id}", va="bottom", color="black")

    vectors = cell_result["edge_gradients"]
    max_norm = max(float(np.linalg.norm(v)) for v in vectors)
    final_norm = float(np.linalg.norm(cell_result["cell_gradient"]))
    scale_reference = max(max_norm, final_norm, 1.0)
    cell_size = max(np.ptp(tri_xy[:, 0]), np.ptp(tri_xy[:, 1]), 1.0)
    scale = 0.35 * cell_size / scale_reference

    for item in cell_result["contributions"]:
        g = item["gradient"]
        arrow = scale * g
        ax.arrow(
            bary[0],
            bary[1],
            arrow[0],
            arrow[1],
            head_width=0.12,
            length_includes_head=True,
            color="0.35",
            alpha=0.85,
        )
        tip = bary + arrow
        ax.text(tip[0], tip[1], f"  E{item['iedge']}", color="0.25", va="center")

    final_arrow = scale * cell_result["cell_gradient"]
    ax.arrow(
        bary[0],
        bary[1],
        final_arrow[0],
        final_arrow[1],
        head_width=0.18,
        linewidth=2.0,
        length_includes_head=True,
        color="crimson",
        label="averaged cell gradient",
    )
    final_tip = bary + final_arrow
    ax.text(final_tip[0], final_tip[1], "  g_cell", color="crimson", weight="bold", va="center")

    ax.text(
        0.02,
        0.98,
        f"C{cell_id}\n"
        f"Nvalid={cell_result['nvalid']}\n"
        f"g=({cell_result['cell_gradient'][0]:.4g}, {cell_result['cell_gradient'][1]:.4g})",
        transform=ax.transAxes,
        ha="left",
        va="top",
        bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "0.6", "alpha": 0.95},
    )

    ax.axhline(0.0, linewidth=0.8, color="0.5")
    ax.axvline(0.0, linewidth=0.8, color="0.5")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("global x")
    ax.set_ylabel("global y")
    ax.set_title(f"Cell C{cell_id}: reconstructed slip-gradient vector")
    ax.grid(True, linewidth=0.5)
    ax.legend(loc="best")
    return fig, ax


fig, ax = plot_cell_slip_gradient(points, cells, selected_cell_id, selected_cell_result)
plt.show()


## Next step: all-cell gradient reconstruction

The notebook now reconstructs a global slip-gradient vector for the central cell `C1` by averaging the valid internal-edge gradient vectors expressed in the common global `(x, y)` frame.

The next algorithmic step will be to call `compute_cell_slip_gradient` for every triangular cell, skipping boundary-edge entries as non-computed observations and reporting `Nvalid` for each cell. Inverse-distance weighting is deliberately left for later.
